In [1]:
# Copyright 2024 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Getting Started with Vertex AI Gemini API

| | |
|-|-|
|Original Authors | [Paul Leroy](https://github.com/paulleroyza) |

## Introduction to the Google GenAI Python Library

The **Google GenAI Python Library** is a client library designed to provide **easy, idiomatic access** to Google's powerful generative AI models, such as the Gemini family.

---

### Primary Use Case

Its primary use case is to help developers and data scientists integrate **advanced generative AI capabilities** directly into their Python applications. This includes:

* **Generating text** (e.g., articles, summaries, creative writing).
* **Generating code** and offering code completion/explanation.
* **Processing multi-modal inputs** (e.g., text, images, and video) to generate complex, relevant outputs.
* Building sophisticated applications like **chatbots, reasoning engines, and data analysis tools** that leverage the models' understanding and generation abilities.

Essentially, it serves as the **simplest interface** for building AI-powered features using Google's foundational models. 

There are many tools that have additional capabilities, but often the applications only need very simple integrations.

## Advantages of the Low-Level GenAI Library Approach

Using the core Google GenAI Python Library offers significant advantages, particularly for developers who need **fine-grained control** over their generative AI applications. Unlike higher-level frameworks that often abstract away complexity, this low-level approach grants the developer **direct access** to model parameters and capabilities, allowing for deep customization.

---

### Direct Control and Customization

A key benefit is the ability to directly manage foundational concepts like **memory and tooling** (function calling). When you use this library, you explicitly handle the state of a conversation (memory) by manually passing the history back and forth in API calls. Similarly, you define and manage the model's access to external tools. This direct control is crucial for building highly specialized applications where standard chat memory or tool usage might be insufficient. For example, a developer can implement custom memory mechanisms, such as a summary-based memory or one that selectively recalls only specific past interactions, leading to more **efficient and contextually relevant** responses.

---

### Understanding and Debugging

Working at this lower level provides an invaluable educational advantage. By seeing the mechanics of how **prompt construction, context injection, and grounding** are manually implemented—whether through system instructions or augmenting the user's input—a developer gains a deeper understanding of the entire generative process. This knowledge becomes a **significant asset** when transitioning to more advanced, automated libraries (like LangChain or Agent Development Kit). When issues arise in those high-level frameworks, the developer understands *what* pieces are being automated and *how* to refine or override the default behavior, transforming a "black box" into a **transparent, modifiable system**. This foundational understanding ultimately leads to **more robust applications and faster debugging**.

## Objectives

This lab provides an introductory understanding of the **Google GenAI Python Library** components and focuses on low-level API mechanics for deep customization and control over model interactions.

  * Mastering the **GenAI API** fundamentals for direct model interaction.
  * Exploring various implementations of **Grounding** using system instructions and context injection.
  * Detailed examination of manually managing **Chat Memory and Grounding** within conversational contexts.
  * Investigating the underlying **Search Principles** that enable the model to integrate real-time, external information (Google Search Grounding).

-----

**References:**

  * [Google GenAI Python Library Documentation](https://ai.google.dev/gemini-api/docs/api-key)
  * [Introduction to the Gemini API](https://ai.google.dev/gemini-api/docs/get-started/python)
  * [GenAI PyPI](https://pypi.org/project/google-genai/)
  * [Generative AI Studio Documentation (Grounding)](https://cloud.google.com/vertex-ai/docs/generative-ai/grounding/overview)

### Costs

This tutorial uses billable components of Google Cloud:

- Vertex AI

Learn about [Vertex AI pricing](https://cloud.google.com/vertex-ai/pricing),
and use the [Pricing Calculator](https://cloud.google.com/products/calculator/)
to generate a cost estimate based on your projected usage.


In [2]:
# Install Vertex AI SDK, LangChain and dependencies
%pip install --upgrade google-genai

Note: you may need to restart the kernel to use updated packages.


In [3]:
# Automatically restart kernel after installs so that your environment can access the new packages
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

### Import libraries
### Set your Google Cloud project ID in the next cell.


In [2]:
PROJECT_ID = "qwiklabs-gcp-02-92710d90f22e"  # @param {type:"string"}
REGION = "us-central1"  # @param {type:"string"}

Next, import the GenAI library and the ipython helpers. Generative models usually respond in markdown so the response can be neatly formatted. It also makes it easy to represent lists, and tables very simply. The project ID and region need to be set for the remote model and the model needs to be chosen. As this will be a chat service for the bulk of this lab a low latency model will be used. It is faster and cheaper but does not have the largest body of knowledge. During this lab the model's knowledge will be augmented with several grounding techniques so the model does need to be that knowledgable.

The gemini-2.5-flash-lite model has a knowledge cutoff of January 2025.

In [3]:
from google import genai
from IPython.display import display, Markdown, clear_output

client = genai.Client(
    vertexai=True, project=PROJECT_ID, location=REGION
)

MODEL_ID='gemini-2.5-flash-lite' 

chat_history= []

def chat(user_input):
    global chat_history
        

    user_message = genai.types.Content(
        role="user",
        parts=[genai.types.Part(text=user_input)]
    )
    
    chat_history.append(user_message)
    
    response = client.models.generate_content(
        model=MODEL_ID,
        contents = chat_history
    )
    
    # Extract the model's response from the result
    model_response_content = response.candidates[0].content
    model_text = model_response_content.parts[0].text
    
    display(Markdown(f"**Gemini:** {model_text}"))
    
    # 3. Append the model's response to our history list
    chat_history.append(model_response_content)

In [4]:
chat("hi")

**Gemini:** Hi there! How can I help you today?

Lets review the model cutoff and limitations. By default the model has no memory, no access to information after its cutoff date.

In [5]:
chat("What is today's weather in Maidenhead?")

**Gemini:** I am sorry, I cannot provide real-time weather information. My knowledge cutoff is **June 2024**, and I am unable to access live data feeds.

To get the most up-to-date weather for Maidenhead, I recommend checking a reliable weather website or app. Here are a few popular options:

*   **BBC Weather:** [https://www.bbc.co.uk/weather/2637389](https://www.bbc.co.uk/weather/2637389)
*   **Met Office:** [https://www.metoffice.gov.uk/weather/forecast/maidenhead](https://www.metoffice.gov.uk/weather/forecast/maidenhead)
*   **AccuWeather:** [https://www.accuweather.com/en/gb/maidenhead/lg4/weather-forecast](https://www.accuweather.com/en/gb/maidenhead/lg4/weather-forecast)

Just open one of these links on your device, and you'll get the latest forecast!

The below cell will usually give the answer for the 2024 Superbowl, sometimes it is aware that it may assume it does not have the latest information and responds accordingly. The answer should be something like:

> The **Kansas City Chiefs** won Super Bowl LVIII. They defeated the San Francisco 49ers.

In [6]:
chat("Who won the superbowl?")

**Gemini:** To tell you who won the Super Bowl, I need to know **which Super Bowl** you're interested in. There's a Super Bowl game every year!

Could you please specify the year (e.g., "Who won the Super Bowl in 2023?" or "Who won the most recent Super Bowl?")?

The `chat_history` list contains all the chat information and is the first part of the grounding capability. The `chat_history` is the part that is passed in it's entirety to the model during each call and can be augemnted before the LLM is called. Also, this is what gives the appearance of the model learning during each turn while in fact it is just reading the entire chat and modifying the next output. To demonstrate this a chat with a name can be entered, and then later interrogated. this lab will only make use of short term memory (chat), but there are options for long term memory as well.

In [8]:
chat("My name is Alan Turing")

**Gemini:** That's a very famous name! Alan Turing was a brilliant mathematician, logician, cryptanalyst, and computer scientist. He's widely considered the father of theoretical computer science and artificial intelligence.

Is there anything specific you'd like to discuss or know about Alan Turing? Perhaps his work on the Enigma code, his contributions to computing, or his personal life?

In [9]:
chat("What is my name?")

**Gemini:** Your name is **Alan Turing**. You've told me this twice now!

The `chat_history` can be reviewed and stored for later interaction. This also means a specific chat can be stored for later use.

In [10]:
chat_history

[Content(
   parts=[
     Part(
       text='hi'
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       text='Hi there! How can I help you today?'
     ),
   ],
   role='model'
 ),
 Content(
   parts=[
     Part(
       text="What is today's weather in Maidenhead?"
     ),
   ],
   role='user'
 ),
 Content(
   parts=[
     Part(
       text="""I am sorry, I cannot provide real-time weather information. My knowledge cutoff is **June 2024**, and I am unable to access live data feeds.
 
 To get the most up-to-date weather for Maidenhead, I recommend checking a reliable weather website or app. Here are a few popular options:
 
 *   **BBC Weather:** [https://www.bbc.co.uk/weather/2637389](https://www.bbc.co.uk/weather/2637389)
 *   **Met Office:** [https://www.metoffice.gov.uk/weather/forecast/maidenhead](https://www.metoffice.gov.uk/weather/forecast/maidenhead)
 *   **AccuWeather:** [https://www.accuweather.com/en/gb/maidenhead/lg4/weather-forecast](https://www.accuweather

## System Messages for grounding

Let's clear the history and re-write the chat function to add the system message as the first part that is passed to the model. Notice that we specify the model for each call meaning tha tthe model can be changed mid-chat to improve thinking capabilities or improve latency, whichever is required for the responses. While this is not usually done in production it is an important feature. The language models themselves have no memory so the entire chat is passed with each call and therefore using different models is very easy to do.

The GenAI `types` library lets the various parts be set.

In [11]:
from google.genai import types

chat_history= []

def chat(user_input):
    global chat_history

    user_message = genai.types.Content(
        role="user",
        parts=[genai.types.Part(text=user_input)]
    )
    
    chat_history.append(user_message)
    
    system_instruction = "You are a weather bot, the weather in Washington is sunny, the weather in the UK is rainy."

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=chat_history,
        config=types.GenerateContentConfig(
            system_instruction=system_instruction,
            max_output_tokens=10000,  # Call out token output
            temperature=0.3, # Call out temperature setting
        ),
    )
    
    # Extract the model's response from the result
    model_response_content = response.candidates[0].content
    model_text = model_response_content.parts[0].text
    
    display(Markdown(f"**Gemini:** {model_text}"))
    
    # 3. Append the model's response to our history list
    chat_history.append(model_response_content)

In [12]:
chat("What is the weather in the UK?")

ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.', 'status': 'RESOURCE_EXHAUSTED'}}

In [ ]:
chat("What is the weather in the Washington?")

One of the really powerful features of the current generation models is that they are multilingual. This is because the original transformer design and the first model `BERT` was designed to handle translation tasks.

In [ ]:
chat("What is the weather in the Washington? Response in French.")

## External files

The input tokens, or context window for the current generation Gemini models is 1M tokens, to put that into perspective, the longest Harry Potter book is **The Order of the Phoenix** at 257000 words or roughly 300k tokens. That means three of those books can be passed in as the imput context. That is also plenty of space to pass in more information from files, audio, images, video and text.

For the next part a file will be put into Google Cloud Storage and then passed to the model at runtime to ground the responses.

This code is from https://github.com/googleapis/python-genai, a worthwhile place to stop by to deepdive into the library.

In [13]:
%%writefile invention_context.txt
The 'Chrono-Converter' was invented in 2077 by Dr. Aris Thorne.
This device doesn't travel through time, but rather converts temporal energy into usable electrical power.
Its primary components are a flux capacitor, a quantum entanglement core, and a standard toaster chassis.
The main limitation of the Chrono-Converter is its tendency to attract rogue squirrels and cause minor paradoxes, such as socks disappearing from the dryer.

Writing invention_context.txt


The next block creates a bucket and gives the Vertex AI service account access to the bucket. The file is copied across to the bucket last.

** Note: ** If the bucket already exists, you will get an error, just ensure the last line show the copy being completed.

> Completed files 1/1 | 434.0B/434.0B    

In [14]:
BUCKET=f"gs://{PROJECT_ID}-gemini"
PROJECT_NUMBER=!gcloud projects list --filter project_id={PROJECT_ID} --format "value(PROJECT_NUMBER)"
# Create Bucket
!gcloud storage buckets create {BUCKET} || true
#Let Gemini Read the bucket
!gcloud storage buckets add-iam-policy-binding {BUCKET} \
    --member='serviceAccount:service-{PROJECT_NUMBER[0]}@gcp-sa-aiplatform.iam.gserviceaccount.com' \
    --role='roles/storage.objectViewer'
!gcloud storage cp invention_context.txt {BUCKET}

Creating gs://qwiklabs-gcp-02-92710d90f22e-gemini/...
bindings:
- members:
  - projectEditor:qwiklabs-gcp-02-92710d90f22e
  - projectOwner:qwiklabs-gcp-02-92710d90f22e
  role: roles/storage.legacyBucketOwner
- members:
  - projectViewer:qwiklabs-gcp-02-92710d90f22e
  role: roles/storage.legacyBucketReader
- members:
  - serviceAccount:service-23661649091@gcp-sa-aiplatform.iam.gserviceaccount.com
  role: roles/storage.objectViewer
etag: CAI=
kind: storage#policy
resourceId: projects/_/buckets/qwiklabs-gcp-02-92710d90f22e-gemini
version: 1
Copying file://invention_context.txt to gs://qwiklabs-gcp-02-92710d90f22e-gemini/invention_context.txt
  Completed files 1/1 | 434.0B/434.0B                                          


For this call you can call the model directly, this means that there will be no memory of the previous interactions.

In [15]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents = types.Content(
        role='user',
        parts=[
          types.Part.from_text(text="Based on the provided document, what are the main components of the Chrono-Converter?"),
          types.Part.from_uri(
            file_uri =  f'{BUCKET}/invention_context.txt',
            mime_type =  'text/plain',
          )
        ]
    )
)

display(Markdown(response.text))

Based on the document, the main components of the Chrono-Converter are:

*   **A flux capacitor**
*   **A quantum entanglement core**
*   **A standard toaster chassis**

## Tools: Google Search

The Gemini API allows you to use some default tools, one of them in Gogole Search. This means that the model can now get updated information from the internet, effectively grounding the model response from the internet. While this is usually beneficial for real time data the internet is not a perfect source for truth.

In [16]:
from google.genai.types import (
    GenerateContentConfig,
    GoogleSearch,
    HttpOptions,
    Tool,
)

response = client.models.generate_content(
    model=MODEL_ID,
    contents="When is the next total solar eclipse in the United States?",
    config=GenerateContentConfig(
        tools=[
            # Use Google Search Tool
            Tool(google_search=GoogleSearch())
        ],
    ),
)

display(Markdown(response.text))

The next total solar eclipse visible in the United States will occur on March 30, 2033, and will be visible in Alaska. The next total solar eclipse in the contiguous United States will be on August 23, 2044, with totality visible in Montana, North Dakota, and South Dakota at sunset. A more widespread total solar eclipse will cross the continental U.S. on August 12, 2045, spanning from California to Florida.

Lets re-ask the question about the weather in Maidenhead

In [17]:
response = client.models.generate_content(
    model=MODEL_ID,
    contents="What is the weather in Maidenhead?",
    config=GenerateContentConfig(
        tools=[
            # Use Google Search Tool
            Tool(google_search=GoogleSearch())
        ],
    ),
)

display(Markdown(response.text))

The current weather in Maidenhead is partly cloudy with a temperature of 50°F (10°C). The RealFeel® temperature is 47°F (8°C). There is a 0% chance of precipitation. The humidity is 76%.

For the rest of the day, the forecast is for it to remain partly cloudy with temperatures reaching a high of 66°F (19°C) and a low of 40°F (4°C).

## Tools: Google Maps

In [19]:
from google import genai
from google.genai import types
from IPython.display import display, Markdown

client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location="us-central1",  # @param {type:"string"}
)

prompt = "What are the best Italian restaurants within a 15-minute walk from here? Give me the names, rating and addresses"

response = client.models.generate_content(
    model='gemini-2.5-flash-lite',
    contents=prompt,
    config=types.GenerateContentConfig(
        tools=[types.Tool(google_maps=types.GoogleMaps())],
        # Optionally provide the relevant location context (this is in Los Angeles)
        tool_config=types.ToolConfig(retrieval_config=types.RetrievalConfig(
            lat_lng=types.LatLng(
                latitude=51.5220755, longitude=-0.7177022))),
    ),
)

display(Markdown(response.text))

if grounding := response.candidates[0].grounding_metadata:
  if grounding.grounding_chunks:
    print("Google Maps sources:")
    for chunk in grounding.grounding_chunks:
      display(Markdown(f'- [{chunk.maps.title}]({chunk.maps.uri})'))

Here are some of the best Italian restaurants within a 15-minute walk:

*   **Sauce and Flour**: This restaurant has a rating of 4.9 stars and is located at 4A High St, Maidenhead SL6 1QJ, UK.
*   **Storia Maidenhead**: This Italian restaurant has a rating of 4.7 stars and is located at 11 Bridge St, Maidenhead SL6 8LR, UK.
*   **Presto Italian Street Food**: This pizza restaurant boasts a 4.9-star rating and is situated at 2A High St, Maidenhead SL6 1QJ, UK.
*   **Knead Neapolitan Pizza, Maidenhead**: With a 4.8-star rating, this pizza restaurant is located at Unit A, Trinity Place, St Ives Rd, Maidenhead SL6 1SG, UK.
*   **Il Michelangelo Maidenhead**: This Italian restaurant is located at 1 Bridge St, Maidenhead SL6 8LR, UK.
*   **ToMo Tankovna**: This European restaurant has a 4.9-star rating and is located at 2c High St, Maidenhead SL6 1QJ, UK.
*   **Coppa Club Maidenhead**: This Mediterranean restaurant has a 4.3-star rating and is located at 2 Bridge Ave, Maidenhead SL6 1RR, UK.

Google Maps sources:


- [Sauce and Flour](https://maps.google.com/?cid=14861104957998893633)

- [Storia Maidenhead](https://maps.google.com/?cid=72113593125231669)

- [Presto Italian Street Food](https://maps.google.com/?cid=4174190337844020154)

...

- [L’assaggino Kingston](https://maps.google.com/?cid=12991587533650324383)

- [Don Beni](https://maps.google.com/?cid=3934406480707880140)

- [La mia Mamma Notting Hill](https://maps.google.com/?cid=17996215383554488587)

- [La Cantina Del Vino - Farnham Common](https://maps.google.com/?cid=15905576134132988005)

- [il Cortile Italian Deli & Cafe](https://maps.google.com/?cid=8161327368363137493)

## Search

In many situations the datasource should be grounded on trusted data and while the context window of 1 million tokens this is usually not enough to hold all the information we need for the model to respond correctly in large organizations. For that you will need to build the capability for context aware search. The main concept here is to break large documents into chunks and find a way to represent those chunks numerically so that you can see which chunks are like the query the user enters. This is basis for Retrieval Augmented Generation and uses embedding models to convert data (generally text) into vectors. 

You'll use the newer gemini-embedding model which is a multilingual embedding model. The older versions had either an English or multilingual version. The advantage of the multilingual model is that you can now search across documents in different languages using the same tooling.

Lets get the embeddings to two sets of similar words. You'll also need a handler to calculate the vector similarity

These formulas typically describe the similarity or distance between two vectors, $\mathbf{A}$ and $\mathbf{B}$, in an $n$-dimensional space, where $\mathbf{A} = (a_1, a_2, \dots, a_n)$ and $\mathbf{B} = (b_1, b_2, \dots, b_n)$.

#### L1 Distance (Manhattan Distance)

The L1 distance measures the sum of the absolute differences between the components of the vectors. It's often called the Manhattan or Taxicab distance because it's the distance a cab would travel in a city grid., Smaller is closer (better)

$$D_{L1}(\mathbf{A}, \mathbf{B}) = \sum_{i=1}^{n} |a_i - b_i|$$

#### L2 Distance (Euclidean Distance)

The L2 distance is the standard straight-line distance between two points in Euclidean space. It's the square root of the sum of the squared differences of the components, based on the Pythagorean theorem., Smaller is closer (better)

$$D_{L2}(\mathbf{A}, \mathbf{B}) = \sqrt{\sum_{i=1}^{n} (a_i - b_i)^2}$$

#### Cosine Similarity

Cosine similarity measures the cosine of the angle $\theta$ between two vectors. It determines how similar the directions of the vectors are, irrespective of their magnitude. The result is 1 for identical direction, 0 for orthogonal (perpendicular), and -1 for exactly opposite direction. Smaller is closer (better)

$$\text{similarity}(\mathbf{A}, \mathbf{B}) = \cos(\theta) = \frac{\mathbf{A} \cdot \mathbf{B}}{||\mathbf{A}|| \cdot ||\mathbf{B}||} = \frac{\sum_{i=1}^{n} a_i b_i}{\sqrt{\sum_{i=1}^{n} a_i^2} \sqrt{\sum_{i=1}^{n} b_i^2}}$$

#### Dot Product (Scalar Product) 

The Dot product (or scalar product) is a measure of the projection of one vector onto another. It is used as a similarity measure, especially when vectors are already normalized (i.e., their L2 norm is 1), in which case it is equivalent to Cosine Similarity. Bigger is closer (better)

$$\text{similarity}(\mathbf{A}, \mathbf{B}) = \mathbf{A} \cdot \mathbf{B} = \sum_{i=1}^{n} a_i b_i$$

In [20]:
import numpy as np

def vector_distance(vec1, vec2, distance_type='l2'):
    """
    Calculates the distance or dot product between two vectors using various metrics.

    Args:
        vec1 (numpy.ndarray): The first vector.
        vec2 (numpy.ndarray): The second vector.
        distance_type (str, optional): The type of calculation to perform. 
                                      Options are 'l1', 'l2', 'cosine', 'euclidean', or 'dot'. 
                                      Defaults to 'l2'.

    Returns:
        float: The calculated distance or dot product.

    Raises:
        ValueError: If the distance_type is invalid.
        ValueError: If the input vectors are not of the same shape (except for 'dot' product).
    """

    vec1 = np.array(vec1)
    vec2 = np.array(vec2)

    if distance_type != 'dot' and vec1.shape != vec2.shape:
        raise ValueError("Input vectors must have the same shape for distance calculations.")

    if distance_type == 'l1':
        return np.sum(np.abs(vec1 - vec2))
    elif distance_type == 'l2' or distance_type == 'euclidean':
        return np.linalg.norm(vec1 - vec2)
    elif distance_type == 'cosine':
        dot_product = np.dot(vec1, vec2)
        norm_vec1 = np.linalg.norm(vec1)
        norm_vec2 = np.linalg.norm(vec2)
        if norm_vec1 == 0 or norm_vec2 == 0:
            return np.nan  # Handle zero vectors.
        return 1 - (dot_product / (norm_vec1 * norm_vec2))
    elif distance_type == 'dot':
        return np.dot(vec1, vec2)
    else:
        raise ValueError("Invalid distance_type. Choose from 'l1', 'l2', 'cosine', 'euclidean', or 'dot'.")

def vec_sim(vec_a,vec_b):
  return {
    "l1_dist": vector_distance(vec_a, vec_b, 'l1'), # Manhattan, smaller is better
    "l2_dist": vector_distance(vec_a, vec_b, 'l2'), # Euclidean, smaller is better (multidimensional pythogoras)
    "cosine_dist": vector_distance(vec_a, vec_b, 'cosine'), # Cosine, smaller is better
    "dot_product": vector_distance(vec_a, vec_b, 'dot') # Dot, bigger is better (higher correlation)
  }

In [21]:
client = genai.Client(
    vertexai=True, project=PROJECT_ID, location=REGION
)

response = client.models.embed_content(
  model="gemini-embedding-001",
  contents=[
    "Supercalifragilisticexpialidocious",
    "Supercalifragilisticexpialidocous", # Misspelt
    "Pneumonoultramicroscopicsilicovolcanoconiosis.", # Longest word in English with a `.` 
    "Pneumonoultramicroscopicsilicovolcanoconiosis",  # Fear of Longest words: hippopotomonstrosesquippedaliophobia
  ],
)

embeddings = response.embeddings

for embedding in embeddings:
  vector = embedding.values
  print (len(vector))
  print (vector[:10])

3072
[-0.0269975196570158, 0.016759516671299934, -0.008210763335227966, -0.0896650031208992, 0.0034660811070352793, -0.003517115954309702, 0.026784798130393028, 0.032589565962553024, 0.003558841533958912, -0.007242843974381685]
3072
[-0.02891998551785946, 0.02274063229560852, 0.004399316385388374, -0.09493998438119888, 0.005439187400043011, -0.002621398074552417, 0.024494057521224022, 0.012268081307411194, 9.11323877517134e-05, -0.0026485950220376253]
3072
[0.0196237713098526, 0.002322945510968566, 0.011539154686033726, -0.07677803188562393, 0.0029539389070123434, 0.012112243101000786, 0.01869363710284233, -0.010011131875216961, -0.010787042789161205, -0.001131572062149644]
3072
[0.015613025985658169, 0.005238987505435944, 0.015012545511126518, -0.07530323415994644, -0.0038251427467912436, 0.010964781045913696, 0.011615362949669361, -0.010863768868148327, -0.01139785349369049, -0.0009703473770059645]


In [22]:
data = []
data.append(vec_sim(embeddings[0].values,embeddings[1].values))
data.append(vec_sim(embeddings[1].values,embeddings[2].values))
data.append(vec_sim(embeddings[2].values,embeddings[3].values))
data.append(vec_sim(embeddings[3].values,embeddings[0].values))
data

[{'l1_dist': np.float64(14.5956511825554),
  'l2_dist': np.float64(0.3362718093468157),
  'cosine_dist': np.float64(0.05653936789160874),
  'dot_product': np.float64(0.9434605818658159)},
 {'l1_dist': np.float64(35.90725177301431),
  'l2_dist': np.float64(0.8235957539064793),
  'cosine_dist': np.float64(0.3391550040209629),
  'dot_product': np.float64(0.660844954876171)},
 {'l1_dist': np.float64(6.519016333714717),
  'l2_dist': np.float64(0.15049058992673145),
  'cosine_dist': np.float64(0.01132370922746695),
  'dot_product': np.float64(0.9886762559164521)},
 {'l1_dist': np.float64(36.68506957920596),
  'l2_dist': np.float64(0.8389468225143449),
  'cosine_dist': np.float64(0.35191589476284124),
  'dot_product': np.float64(0.6480840881851866)}]

In [23]:
# Super cal... + mis-spelt
vec_sim(embeddings[0].values,embeddings[1].values)

{'l1_dist': np.float64(14.5956511825554),
 'l2_dist': np.float64(0.3362718093468157),
 'cosine_dist': np.float64(0.05653936789160874),
 'dot_product': np.float64(0.9434605818658159)}

In [24]:
# Supercala vs Pneumo
vec_sim(embeddings[1].values,embeddings[2].values)

{'l1_dist': np.float64(35.90725177301431),
 'l2_dist': np.float64(0.8235957539064793),
 'cosine_dist': np.float64(0.3391550040209629),
 'dot_product': np.float64(0.660844954876171)}

In [25]:
# Pneumo .. vs Pneumo.
vec_sim(embeddings[2].values,embeddings[3].values)

{'l1_dist': np.float64(6.519016333714717),
 'l2_dist': np.float64(0.15049058992673145),
 'cosine_dist': np.float64(0.01132370922746695),
 'dot_product': np.float64(0.9886762559164521)}

In [26]:
# Last Pneumo vs first Supercala ..
vec_sim(embeddings[3].values,embeddings[0].values)

{'l1_dist': np.float64(36.68506957920596),
 'l2_dist': np.float64(0.8389468225143449),
 'cosine_dist': np.float64(0.35191589476284124),
 'dot_product': np.float64(0.6480840881851866)}

You can compare the user query with a document chunk and calculate the vector similarity.

In [27]:
response = client.models.embed_content(
  model="gemini-embedding-001",
  contents=[
    "What was Alphabet's net income in 2022?",
    "Alphabet's net income in 2022 was 59,972.",
  ],
)

embeddings = response.embeddings

vec_sim(embeddings[0].values,embeddings[1].values)

{'l1_dist': np.float64(18.255991876423195),
 'l2_dist': np.float64(0.4160082547193056),
 'cosine_dist': np.float64(0.08653142494837185),
 'dot_product': np.float64(0.9134686705765971)}

The embedding vectors require specialised storage capable of both storing the vectors and search for similar vectors. All the managed databases in Google Cloud have this capability catering for broad requirements across the organization. In addition there is the Vector Search service built specifically for this purpose. While out of scope for this lab you now understand the basics of being able to find relevant information that can be passed to the model for grounding.